# D-MPNN training using chemprop v2

Chemprop's Directed Message Passing Neural Network (D-MPNN) is the standard
published benchmark architecture for molecular property prediction (Yang et
al. 2019). It is trained here on the same scaffold split and same pIC50
target as the classical descriptor models and the from scratch GNN, so its
scores are directly comparable to theirs. Using a well tested, widely
adopted library implementation (rather than only a custom from scratch GNN)
gives a credible upper bound reference for "how good can a graph model get
on this dataset", and any gap between this and the from scratch GNN is a
useful signal about implementation/engineering quality versus architecture
choice.

In [ ]:
from pathlib import Path
import pandas as pd
import wandb
from lightning import pytorch as pl
from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint
import chemprop
from chemprop import data, featurizers, models, nn


### 1. Load scaffold split

Reusing the same `scaffold_train`/`scaffold_val`/`scaffold_test` CSVs as the
classical ML and from scratch GNN notebooks.

In [ ]:
data_path = Path.cwd().parent
splits_dir = data_path / "data/splits"

In [ ]:
smiles_column = "smiles"
target_column = "pIC50"
train_df = pd.read_csv(splits_dir/"scaffold_train.csv")
val_df = pd.read_csv(splits_dir/"scaffold_val.csv")
test_df = pd.read_csv(splits_dir/"scaffold_test.csv")

train_df.shape, val_df.shape, test_df.shape

((8401, 5), (1050, 5), (1051, 5))

### 2. Build Molecule Datapoints, MoleculeDataset and DataLoader

Chemprop needs SMILES converted into its own graph representation before
training: a `MoleculeDatapoint` pairs one molecule's SMILES with its target
value, a `MoleculeDataset` batches many datapoints together with a shared
featurizer, and a `DataLoader` handles the actual minibatching during
training.

In [4]:
def df_to_datapoints(df, smiles_col, target_col):
    """Convert a dataframe of SMILES/target pairs into chemprop MoleculeDatapoints.

    `MoleculeDatapoint.from_smi` parses each SMILES string into an RDKit mol
    internally and stores the target(s) alongside it; the actual graph
    featurization (atoms/bonds -> tensors) happens later, lazily, via the
    featurizer passed to MoleculeDataset.
    """
    smiles = df[smiles_col].values
    target = df[[target_col]].values
    return [data.MoleculeDatapoint.from_smi(smi, y) for smi, y in zip(smiles, target)]

train_data = df_to_datapoints(train_df, smiles_column, target_column)
val_data = df_to_datapoints(val_df, smiles_column, target_column)
test_data = df_to_datapoints(test_df, smiles_column, target_column)


In [ ]:
featurizer = featurizers.SimpleMoleculeMolGraphFeaturizer()
train_dataset = data.MoleculeDataset(train_data, featurizer)
# Fit the target scaler on TRAIN ONLY (never val/test) to avoid leaking information.
scaler = train_dataset.normalize_targets()

val_dataset = data.MoleculeDataset(val_data, featurizer)
val_dataset.normalize_targets(scaler)

test_dataset = data.MoleculeDataset(test_data, featurizer)

train_loader = data.build_dataloader(train_dataset, num_workers=0)
# shuffle=False for val/test so predictions can be matched back to rows by
# position later, without needing to track a shuffled index mapping.
val_loader = data.build_dataloader(val_dataset, num_workers = 0, shuffle = False)
test_loader = data.build_dataloader(test_dataset, num_workers = 0, shuffle = False)


In [ ]:
train_loader


In [ ]:
batch = next(iter(train_loader))
print(type(batch))
print(batch._fields)

<class 'chemprop.data.collate.TrainingBatch'>
('bmg', 'V_d', 'X_d', 'Y', 'w', 'lt_mask', 'gt_mask')


In [8]:
# Unpack the named fields of a TrainingBatch:
#   bmg          - BatchMolGraph: the batched atom/bond graph tensors chemprop's
#                  message passing network actually consumes
#   V_d, X_d     - optional extra atom/molecule level descriptor features (unused here,
#                  since this run relies purely on the learned graph representation)
#   Y            - regression targets (scaled pIC50)
#   w            - per sample loss weights
#   lt_mask/gt_mask - masks for censored ("less than"/"greater than") labels, not used
#                  for this dataset since all pIC50 values are exact measurements
bmg, V_d, X_d, Y, weights, lt_mask, gt_mask = batch

print("BatchMolGraph:", bmg)
print("Y shape:", Y.shape)        # (batch_size, 1) — your pIC50 targets, scaled
print("Y sample:", Y[:5])
print("weights:", weights[:5])    # sample weights, usually all 1s unless you set them

BatchMolGraph: <chemprop.data.collate.BatchMolGraph object at 0x1362afec0>
Y shape: torch.Size([64, 1])
Y sample: tensor([[ 1.1174],
        [-0.5700],
        [-1.2376],
        [-1.4797],
        [-1.2743]])
weights: tensor([[1.],
        [1.],
        [1.],
        [1.],
        [1.]])


### 3. Build the MPNN

D-MPNN passes learned "messages" along bonds (rather than atoms, as in a
vanilla message passing GNN) for several rounds, then pools atom
representations into one molecule level vector for the regression head.
This lets the network learn its own task specific molecular representation
directly from the graph, instead of relying on a fixed, hand engineered
descriptor like ECFP fingerprints.

In [ ]:
# BondMessagePassing is the "D" (directed, bond centric) in D-MPNN: it
# avoids the immediate reversal problem of atom centric message passing (a
# message bouncing straight back along the bond it just came from).
mp = nn.BondMessagePassing()

# Pool per atom learned vectors into one fixed size molecule vector by mean
# rather than sum, so molecule size doesn't dominate the pooled representation.
agg = nn.MeanAggregation()

# Output transform to unscale predictions back to real pIC50 units, 
output_transform = nn.UnscaleTransform.from_standard_scaler(scaler)

ffn = nn.RegressionFFN(output_transform=output_transform)

metric_list = [nn.metrics.RMSE(), nn.metrics.MAE()]

batch_norm = True

mpnn = models.MPNN(mp, agg, ffn, batch_norm, metric_list)
mpnn

MPNN(
  (message_passing): BondMessagePassing(
    (W_i): Linear(in_features=86, out_features=300, bias=False)
    (W_h): Linear(in_features=300, out_features=300, bias=False)
    (W_o): Linear(in_features=372, out_features=300, bias=True)
    (dropout): Dropout(p=0.0, inplace=False)
    (tau): ReLU()
    (V_d_transform): Identity()
    (graph_transform): Identity()
  )
  (agg): MeanAggregation()
  (bn): BatchNorm1d(300, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (predictor): RegressionFFN(
    (ffn): MLP(
      (0): Sequential(
        (0): Linear(in_features=300, out_features=300, bias=True)
      )
      (1): Sequential(
        (0): ReLU()
        (1): Dropout(p=0.0, inplace=False)
        (2): Linear(in_features=300, out_features=1, bias=True)
      )
    )
    (criterion): MSE(task_weights=[[1.0]])
    (output_transform): UnscaleTransform()
  )
  (X_d_transform): Identity()
  (metrics): ModuleList(
    (0): RMSE(task_weights=[[1.0]])
    (1): MAE

### 4. Set up and run the trainer

Uses PyTorch Lightning's `Trainer` to handle the training loop, checkpointing
of the best model by validation loss, and early stopping — the same
supervised regression setup as the from scratch GNN notebook, so training
protocol differences don't confound the model comparison later.

In [ ]:
# Save the checkpoint with the lowest validation loss 
checkpointing = ModelCheckpoint(
    "models/chemprop_checkpoints",
    "best-{epoch}-{val_loss:.2f}",
    "val_loss",
    mode="min",
    save_last=True,
)

# patience=10 matches the dmpnn/chemeleon tracks (scripts/train_chemprop.py) for a fair comparison
early_stopping = EarlyStopping(monitor="val_loss", mode="min", patience=10)

trainer = pl.Trainer(
    logger=False,          # W&B logging is handled manually below instead
    enable_checkpointing=True,
    enable_progress_bar=True,
    accelerator="auto",    # use GPU/MPS automatically if available, else CPU
    devices=1,
    max_epochs=50,          # upper bound; early stopping usually triggers first
    callbacks=[checkpointing, early_stopping],
)


GPU available: False, used: False
TPU available: False, using: 0 TPU cores


In [ ]:
run = wandb.init(
    project="egfr-pic50-prediction",
    job_type="chemprop",
    group="scaffold",
    name="chemprop_baseline_scaffold_split",
    config={
        "split_type": "scaffold",
        "descriptor_type": "learned_graph",
        "model": "chemprop_MPNN_regression",
        "message_passing": "BondMessagePassing",
        "aggregation": "mean",
        "batch_norm": True,
        "max_epochs": 50,
        "early_stopping_patience": 10,
        "train_size": len(train_dataset),
        "val_size": len(val_dataset),
        "test_size": len(test_dataset),
    },
)

trainer.fit(mpnn, train_loader, val_loader)

val_results = trainer.validate(dataloaders=val_loader, weights_only=False)[0]
wandb.log({f"val_{k}": v for k, v in val_results.items()})

artifact = wandb.Artifact("chemprop-default-scaffold", type="model")
artifact.add_file(str(checkpointing.best_model_path))
run.log_artifact(artifact)
wandb.finish()

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /Users/ankitkumar/.netrc.
wandb: Currently logged in as: ankitchem3 (ankitchem3-self-employed) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


/opt/miniconda3/envs/egfr-env/lib/python3.11/site-packages/lightning/pytorch/callbacks/model_checkpoint.py:881: Checkpoint directory /Users/ankitkumar/Documents/Projects/egfr-pic50-prediction/notebooks/models/chemprop_checkpoints exists and is not empty.
Loading `train_dataloader` to estimate number of stepping batches.
/opt/miniconda3/envs/egfr-env/lib/python3.11/site-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/opt/miniconda3/envs/egfr-env/lib/python3.11/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.


   | Name            | Type               | Params | Mode  | FLOPs
-----------------------------------------------------------------------
0  | message_passing | BondMessagePassing |  227 K | train |     0
1  | agg             | MeanAggregation    |      0 | train |     0
2  | bn              | BatchNorm1d        |    600 | train |     0
3  | predictor       | RegressionFFN      | 90.6 K | train |     0
4  | X_d_transform   | Identity           |      0 | train |     0
5  | metrics         | ModuleList         |      0 | train |     0
-----------------------------------------------------------------------
Trainable params: 318 K
Non-trainable params: 0
Total params: 318 K
Total estimated model params size (MB): 1.276
Modules in train mode: 25
Modules in eval mode: 0
Total FLOPs: 0


/opt/miniconda3/envs/egfr-env/lib/python3.11/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.
/opt/miniconda3/envs/egfr-env/lib/python3.11/site-packages/lightning/pytorch/trainer/connectors/checkpoint_connector.py:149: `.validate(ckpt_path=None)` was called without a model. The best model of the previous `fit` call will be used. You can pass `.validate(ckpt_path='best')` to use the best model or `.validate(ckpt_path='last')` to use the last model. If you pass a value, this warning will be silenced.
Restoring states from the checkpoint path at /Users/ankitkumar/Documents/Projects/egfr-pic50-prediction/notebooks/models/chemprop_checkpoints/best-epoch=17-val_loss=0.50.ckpt
Loaded model weights from the checkpoint at /Users/ankitkumar/Documents/Projects/egfr-pic50-predi

     Validate metric           DataLoader 0
---------------------------------------------------
        val/mae               0.5336832404136658
        val/rmse              0.7042626142501831
        val_loss              0.49598586559295654


In [12]:
# Evaluate on the held out scaffold test set using the best checkpoint
# (ckpt_path="best"), not the final epoch weights, for the headline numbers.
test_results = trainer.test(dataloaders=test_loader, ckpt_path="best", weights_only=False)[0]
print(test_results)

Restoring states from the checkpoint path at /Users/ankitkumar/Documents/Projects/egfr-pic50-prediction/notebooks/models/chemprop_checkpoints/best-epoch=17-val_loss=0.50.ckpt
Loaded model weights from the checkpoint at /Users/ankitkumar/Documents/Projects/egfr-pic50-prediction/notebooks/models/chemprop_checkpoints/best-epoch=17-val_loss=0.50.ckpt
/opt/miniconda3/envs/egfr-env/lib/python3.11/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'test_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.


     Test metric             DataLoader 0
---------------------------------------------------
        test/mae              0.7319768071174622
        test/rmse             0.9723594188690186


{'test/rmse': 0.9723594188690186, 'test/mae': 0.7319768071174622}


In [ ]:
from sklearn.metrics import r2_score
from scipy.stats import spearmanr
import numpy as np
import torch

predictions = trainer.predict(dataloaders=test_loader, ckpt_path="best", weights_only=False)

# predictions is a list of batch tensors — concatenate into one array
preds = torch.cat(predictions).numpy().flatten()
true_vals = test_df["pIC50"].values


rmse = np.sqrt(np.mean((preds - true_vals) ** 2))
mae = np.mean(np.abs(preds - true_vals))
r2 = r2_score(true_vals, preds)
spearman_rho, _ = spearmanr(true_vals, preds)

print(f"Test RMSE: {rmse:.4f}")
print(f"Test MAE: {mae:.4f}")
print(f"Test R²: {r2:.4f}")
print(f"Test Spearman ρ: {spearman_rho:.4f}")

Restoring states from the checkpoint path at /Users/ankitkumar/Documents/Projects/egfr-pic50-prediction/notebooks/models/chemprop_checkpoints/best-epoch=17-val_loss=0.50.ckpt
Loaded model weights from the checkpoint at /Users/ankitkumar/Documents/Projects/egfr-pic50-prediction/notebooks/models/chemprop_checkpoints/best-epoch=17-val_loss=0.50.ckpt
/opt/miniconda3/envs/egfr-env/lib/python3.11/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'predict_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.


Test RMSE: 0.9724
Test MAE: 0.7320
Test R²: 0.4219
Test Spearman ρ: 0.6760


In [ ]:
val_predictions = trainer.predict(dataloaders=val_loader, ckpt_path="best", weights_only=False)
val_preds = torch.cat(val_predictions).numpy().flatten()
val_resid_std = float(np.std(val_df["pIC50"].values - val_preds))


out = test_df.copy()
out["pIC50"] = preds
out["pIC50_unc"] = val_resid_std

out_path = data_path / "models" / "chemprop" / "scaffold" / "test_predictions.csv"
out.to_csv(out_path, index=False)
print(f"wrote {len(out)} predictions -> {out_path}")

Restoring states from the checkpoint path at /Users/ankitkumar/Documents/Projects/egfr-pic50-prediction/notebooks/models/chemprop_checkpoints/best-epoch=17-val_loss=0.50.ckpt
Loaded model weights from the checkpoint at /Users/ankitkumar/Documents/Projects/egfr-pic50-prediction/notebooks/models/chemprop_checkpoints/best-epoch=17-val_loss=0.50.ckpt
/opt/miniconda3/envs/egfr-env/lib/python3.11/site-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/opt/miniconda3/envs/egfr-env/lib/python3.11/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'predict_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.


wrote 1051 predictions -> /Users/ankitkumar/Documents/Projects/egfr-pic50-prediction/models/chemprop/scaffold/test_predictions.csv
